In [6]:
import pandas as pd
from textblob import TextBlob

In [7]:
df = pd.read_csv('../data/comments.csv')
df2 = pd.read_csv('../data/statistics.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   video_id  143 non-null    object
 1   comments  142 non-null    object
 2   date      143 non-null    object
 3   time      143 non-null    object
dtypes: object(4)
memory usage: 4.6+ KB


In [8]:

def comments_sentiment(text):
    blob = TextBlob(str(text))
    polarity = blob.sentiment.polarity
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    else:
        return 'neutral'
    
df['comments'] = df['comments'].apply(comments_sentiment)
df.to_csv('../data/comments_sentiments.csv', index=False)
display(df)

,video_id,comments,date,time
0,zsjvFFKOm3c,negative,2021-04-06,14:47:25
1,zsjvFFKOm3c,neutral,2021-04-06,14:47:25
2,zsjvFFKOm3c,neutral,2021-04-06,14:47:25
3,zsjvFFKOm3c,negative,2021-04-06,14:47:25
4,zsjvFFKOm3c,neutral,2021-04-06,14:47:25
...,...,...,...,...
138,l8DCPaHc5TQ,positive,2022-08-23,08:00:15
139,l8DCPaHc5TQ,positive,2022-08-23,08:00:15
140,l8DCPaHc5TQ,neutral,2022-08-23,08:00:15
141,l8DCPaHc5TQ,positive,2022-08-23,08:00:15


In [9]:
df3 = pd.read_csv('../data/comments_sentiments.csv')

df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   video_id  143 non-null    object
 1   comments  143 non-null    object
 2   date      143 non-null    object
 3   time      143 non-null    object
dtypes: object(4)
memory usage: 4.6+ KB


In [10]:
def sentiment_percentages(group):
    total = len(group)
    positive = (group == 'positive').sum() / total * 100
    negative = (group == 'negative').sum() / total * 100
    neutral = (group == 'neutral').sum() / total * 100
    return pd.Series({
        'positive_pct': positive,
        'negative_pct': negative,
        'neutral_pct': neutral,
        'total_comments': total
    })


sentiment_summary = df3.groupby('video_id')['comments'].apply(sentiment_percentages).reset_index()

df_merged = df2.merge(sentiment_summary, on='video_id', how='left')

df_merged['engagement_stats'] = (
    df_merged['viewCount'] * 0.1 + df_merged['likeCount'] * 1 + df_merged['commentCount'] * 2
)

df_wide = df_merged.pivot_table(
    index=['video_id', 'channel_title', 'title', 'viewCount', 'likeCount', 'commentCount', 'date', 'time'],
    columns='level_1',
    values='comments'
).reset_index()

cleaned_sentiment = df_wide[df_wide['total_comments'] > 1.0]
cleaned_sentiment = df_wide.drop(columns=['channel_title', 'title'])

cleaned_sentiment.to_csv('../data/sentiment.csv', index=False)
display(cleaned_sentiment.head(100))


level_1,video_id,viewCount,likeCount,commentCount,date,time,negative_pct,neutral_pct,positive_pct,total_comments
0,27axs9dO7AE,2111131,51484,1391,2019-04-16,13:53:16,0.0,60.0,40.0,5.0
1,2i7YhEjo1Jw,103414,8198,23,2023-07-20,12:45:33,20.0,60.0,20.0,5.0
2,3s0lFtUrhSQ,114482,2512,38,2023-07-29,06:57:08,0.0,60.0,40.0,5.0
3,7S_tz1z_5bA,12948455,270389,9772,2019-03-20,00:50:32,20.0,20.0,60.0,5.0
4,7mz73uXD9DA,794690,23458,865,2024-03-11,10:00:11,20.0,60.0,20.0,5.0
5,9ctKDYilxkU,12202,819,2,2024-03-18,14:25:26,0.0,100.0,0.0,2.0
6,Ah_LMYqd2CE,5721078,233245,12955,2025-02-22,00:18:28,20.0,20.0,60.0,5.0
7,Cz3WcZLRaWc,1057320,35199,1528,2021-04-16,15:41:25,0.0,40.0,60.0,5.0
8,HXV3zeQKqGY,19673794,365172,11646,2018-07-02,17:13:32,0.0,60.0,40.0,5.0
9,HZ5eTsH3_TM,180858,17868,61,2023-03-21,00:21:41,0.0,40.0,60.0,5.0
